In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random
from common import *

In [ ]:
set_seed(0)
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})

plt.rcParams.update({
    'text.usetex': False,
    'mathtext.fontset': 'cm',  # Computer Modern math font
})
colors = ["blue", "red", "green"]

default plot

In [ ]:
c1 = 0.5
c2 = 1
def sample_n(batch, sizes, device, mul=0.1):
    k = torch.arange(sizes, device=device).unsqueeze(dim=0).repeat(batch, 1).float()
    b1 = (torch.rand((batch, 1), device=device).round() * 2 - 1)
    b2 = (torch.rand((batch, 1), device=device).round() * 2 - 1)
    x = b1 * c1 * torch.sin(k * 2 * torch.pi / 24) + b2 * c2 * torch.sin(8 * k * 2 * torch.pi / 24)
    epsilon1 = mul * torch.randn_like(x)
    epsilon2 = mul * torch.randn_like(x)
    return x + epsilon1, x + epsilon2

In [ ]:
def get_e(sizes, device):
    k = torch.arange(sizes, device=device).unsqueeze(dim=0).float()
    e1 = c1 * torch.sin(k * 2 * torch.pi / 24)
    e2 = c2 * torch.sin(k * 8 * 2 * torch.pi / 24)
    return torch.stack([e1, e2], dim=0)
eig_order = ["{l}", "{s}"]
e_order = ["{s}", "{l}"]

In [ ]:
sizes = 96
samples = sample_n(1, 96, "cuda")
# plt.xlim(0, sizes-1)
fig, axs = plt.subplots(1, 1, figsize=(14, 3.7))
axs.plot(samples[0][0].cpu(), color=colors[0])
axs.set_xlabel(r"$t$")
axs.set_ylabel(r"$x$")
plt.savefig("figure/freq_input.pdf", format="pdf", bbox_inches = 'tight')
plt.tight_layout()
plt.show()

In [ ]:
batch = 10000
epochs = 500

s_in = sizes
s_out = 2
device = torch.device("cuda")
simple_network = nn.Linear(s_in, s_out, bias=False, device=device)
lr_simple = 1e-3
scaling = 1e-4
with torch.no_grad():
    torch.nn.init.normal_(simple_network.weight)
print(torch.var(simple_network.weight), torch.mean(simple_network.weight))
lossFn = BTLoss(in_feature=s_out, use_batchnorm=False, lambd=1).to(device)

scale_weights(simple_network, scaling)
t = sample_n(batch, sizes, device)
gamma = t[0].T @ t[1]  # n x n
gamma = (gamma + gamma.T) / (2 * batch)
plt.imshow(gamma.cpu())
plt.tight_layout()
plt.colorbar()
plt.show()
gamma_l, gamma_v = torch.linalg.eig(gamma)
gamma_l = gamma_l.real.cpu()
gamma_v = gamma_v.real.cpu()
print(gamma_l, gamma_v)
w = simple_network.weight.detach().clone()
u, s0, g = torch.linalg.svd(w)
plt.imshow(gamma_l.unsqueeze(dim=0))
plt.show()
plt.imshow(gamma_v)
plt.show()

In [ ]:
print(gamma_l[0:4])
fig, axs = plt.subplots(1, 3, figsize=(24, 3.7))
for i in range(3):
    axs[i].plot(gamma_v.T[i].cpu(), color=colors[i])
    axs[i].set_xlabel(r"$v_" + str(i+1) + r"\; \text{index}$")
plt.tight_layout()
plt.savefig(f"figure/freq_gamma_eigenvectors.pdf", format="pdf", bbox_inches = 'tight')
plt.show()

In [ ]:
optimizer = torch.optim.SGD(simple_network.parameters(), lr_simple, momentum=0, weight_decay=0)
simple_network.train()

loss_values = []
loss_values_deep = []
eiglist = [[] for i in range(s_out)]
we = [[] for _ in range(s_out)]
we_deep = [[] for _ in range(s_in)]
ones = torch.ones(s_in, device=device)
zeros = torch.zeros(s_in, device=device)
e = get_e(sizes, device)

weight_singular = [[] for _ in range(s_out)]
plane_alignment = []
uwe = [[[] for _ in range(s_out)] for _ in range(s_out)]

print("e", e)
weight_center = None
eig_reversed = False
weight_first = simple_network.weight.detach().clone()
for epoch in tqdm(range(1, epochs + 1)):
    optimizer.zero_grad()
    x0, x1 = sample_n(batch, sizes, device)
    z0, z1 = torch.chunk(simple_network(torch.cat([x0.clone(), x1.clone()], dim=0)), 2, dim=0)
    c, loss = lossFn(z0, z1)
    c = c.detach()
    if epoch == 1:
        print("output", c)
    v = reversed(torch.linalg.eigvalsh(c))
    for i in range(len(v)):
        eiglist[i].append(v[i].cpu().item())
    loss_values.append(loss.item())
    loss.backward()
    optimizer.step()
    
    
    u_prev = u_w_current if epoch > 1 else None
    w_current = simple_network.weight.detach().clone()
    u_w_current, s_w_current, v_w_current_h = torch.linalg.svd(w_current)
    v_w_current = v_w_current_h.T
    
    if epoch > 1:
        dist1 = abs(F.cosine_similarity(u_prev[:, 0], u_w_current[:, 0], dim=0))
        dist2 = abs(F.cosine_similarity(u_prev[:, 1], u_w_current[:, 0], dim=0))
        eig_reversed = dist1 < dist2

    if eig_reversed:
        u_w_current = u_w_current[:, [1, 0]]
        s_w_current = s_w_current[[1, 0]]
        v_w_current = v_w_current[:, [1, 0]]
    u_prev = u_w_current.clone()
        
    for i in range(s_out):
        weight_singular[i].append(s_w_current[i].cpu().item())
        we[i].append(torch.norm(e[i].unsqueeze(dim=0) @ w_current.T).cpu().item())
    plane_alignment.append((torch.norm(v_w_current[:, :2].T @ gamma_v[:, :2].cuda(), p='fro')**2/2).cpu().item())
    
    for i in range(2):
        for j in range(2):
            uwe[i][j].append(((u_w_current[:, i] @ w_current @ e[j].squeeze().cuda()).cpu()).item())
    
    if abs(loss.item() - 1) < 0.02 and weight_center is None:
        weight_center = simple_network.weight.detach().clone()


In [ ]:
uwf, swf, vwfh = torch.linalg.svd(weight_first)
uwf = uwf.cpu()
swf = swf.cpu()
vwf = vwfh.T.cpu()

uwa, swa, vwha = torch.linalg.svd(weight_center)
uwa = uwa.cpu()
swa = swa.cpu()
vwa = vwha.T.cpu()

fig, axs = plt.subplots(2, 3, figsize=(24, 7.5))
uwList = [vwf, vwa, v_w_current.cpu()]
for t in range(3):
    for i in range(2):
        axs[i][t].plot(uwList[t].T[i], color=colors[i])
        # axs[0][i].set_xlim(0, sizes-1)
        axs[i][t].set_xlabel(r"$v_" + str(i+1) + r"\; \text{index}$")
        # axs[i][t].set_ylabel("Value")
plt.tight_layout()
plt.savefig("figure/freq_v.pdf", format="pdf", bbox_inches = 'tight')
plt.show()

In [ ]:
gamma_0, gamma_1 = gamma_l[0], gamma_l[1]
t0=1490
t = np.linspace(0, epochs, epochs)
y1 = gamma_0 / (gamma_0 + s0[0].cpu()**(-2) * np.exp(-8 * gamma_0 * t/t0))
y2 = gamma_1 / (gamma_1 + s0[1].cpu()**(-2) * np.exp(-8 * gamma_1 * t/t0))
lll = (y1-1)**2 + (y2-1)**2

In [ ]:
palette = plt.rcParams['axes.prop_cycle'].by_key()['color']
alpha = 0.8
fig, axs = plt.subplots(1, 3, figsize=(24, 3.7))

axs[0].plot(loss_values, color=colors[2], linewidth=6, alpha=0.5)
index = []
for i in range(len(eiglist)):
    index.append(((torch.tensor(loss_values) - (s_out - i - 0.5)) < 0).nonzero()[0])
axs[0].plot(lll, linestyle='--', color=colors[2], linewidth=2.5)
axs[0].set_ylim(-0.05, s_out * 1.15)
print(index)
axs[0].vlines(index, -1, 3, colors='black', linestyle='--')
axs[0].set_xlabel("Step")
axs[0].set_ylabel("Loss")

for i in range(len(eiglist)):
    axs[1].plot(eiglist[i], linewidth=6, color=colors[i], alpha=0.5, label=(r"$\lambda_" + eig_order[i] + "$"))
axs[1].plot(t, y1, linestyle='--', color=colors[0], linewidth=2.5)
axs[1].plot(t, y2, linestyle='--', color=colors[1], linewidth=2.5)
axs[1].vlines(index, -1, 3, colors='black', linestyle='--')
axs[1].set_ylim(-0.05, 1.10)
axs[1].set_xlabel("Step")
axs[1].set_ylabel(r"C Eigenvalue $\lambda$")
axs[1].legend()

for i in reversed(range(len(we))):
    k = we[i]
    # for j in range(len(k)):
    #     k[j] /= k[-1]
    axs[2].plot(k, label=(r"${\text{FA}(e_" + e_order[i] + ")}$"), alpha=0.5, linewidth=6, color=colors[1-i])
axs[2].plot(t, np.sqrt(y1), linestyle='--', color=colors[0], linewidth=2.5)
axs[2].plot(t, np.sqrt(y2), linestyle='--', color=colors[1], linewidth=2.5)
axs[2].vlines(index, -1, 3, colors='black', linestyle='--')
axs[2].set_ylim(-0.05, 1.10)
axs[2].legend()
axs[2].set_xlabel("Step")
axs[2].set_ylabel(r"$\|We_i\|_2$")

plt.tight_layout()
plt.savefig("figure/freq_loss_eig_align.pdf", format="pdf", bbox_inches = 'tight')
plt.show()

In [ ]:
##dc/dt, -dl/dt
fig, axs = plt.subplots(1, 1, figsize=(14, 3.7))
for i in reversed(range(2)):
    plt.plot(np.array(dt(eiglist[i])) / lr_simple, alpha=0.5, linewidth=6, color=colors[i], label=r"$d\lambda_" + eig_order[i] + "/dt$")

dlossdt = -1 * np.array(dt(loss_values))
axs.plot(dlossdt / lr_simple, alpha=1, linestyle='--', color='black', linewidth=1.5, label=r"$-d\mathcal{L}/dt$")
# axs.hlines([max(dlossdt) / lr_simple], xmin=-1, xmax=epochs*1.1, color='black', alpha=0.8, linewidth=3)
# axs.hlines([max(dlossdt[1000:]) / lr_simple], xmin=-1, xmax=epochs*1.1, color='black', alpha=0.8, linewidth=3)
axs.hlines([gamma_0 * 2], xmin=-1, xmax=epochs*1.1, color=colors[0], linewidth=3, alpha=0.8, linestyle='--')
axs.hlines([gamma_1 * 2], xmin=-1, xmax=epochs*1.1, color=colors[1], linewidth=3, alpha=0.8, linestyle='--')

axs.set_yticks([-60, 0, 60, 120,])

y_pos = [text.get_position()[1] for text in axs.get_yticklabels()]#[:-1]
y_txt = [text.get_text() for text in axs.get_yticklabels()]#[:-1]

new_pos0 = gamma_0 * 2
new_txt0 = r"$2\gamma_l$"
new_pos1 = gamma_1 * 2
new_txt1 = r"$2\gamma_s$"

y_pos.append(new_pos0)
y_pos.append(new_pos1)
y_txt.append(new_txt0)
y_txt.append(new_txt1)
axs.set_yticks(y_pos,y_txt)

# axs.set_ylim(-0.025, epochs)
axs.set_xlim(0, epochs/2)
axs.set_xlabel("Step")
# axs.set_ylabel("Value")
axs.legend(loc='upper right')
plt.savefig("figure/freq_dcdt_dldt.pdf", format="pdf", bbox_inches = 'tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(22, 3.7))
# t-s_j
for i in reversed(range(2)):
    lambda_i = y1 if i == 0 else y2
    gamma_i = gamma_l[i].item()
    axs[0].plot(weight_singular[i], alpha=0.5, linewidth=6, color=colors[i], label=(r"$s_" + eig_order[i] + "$"))
    
    s_j = np.power(lambda_i / gamma_i, 1/2)
    axs[0].plot(s_j, alpha=0.8, linewidth=4, color=colors[i], linestyle='--')
    a_j = np.power(s_j, -1/2)
    p = (s_j[0]**(-2))*np.exp(-8*gamma_i*t/t0)
    s_j_p_theory = 1/(2*s_j)*(-np.power(gamma_i + p, -2) * (-8 * gamma_i * p))
    axs[1].plot(s_j_p_theory, alpha=0.8, linewidth=4, color=colors[i], linestyle='--')
print(weight_singular[0][-1])
print(weight_singular[1][-1])
axs[0].set_xlim(0, epochs)
axs[0].hlines([1/np.sqrt(gamma_0)], xmin=-1, xmax=epochs*1.1, color=colors[0], linewidth=3, alpha=0.8, linestyle='--')
axs[0].hlines([1/np.sqrt(gamma_1)], xmin=-1, xmax=epochs*1.1, color=colors[1], linewidth=3, alpha=0.8, linestyle='--')
axs[0].set_xlabel("Step")
axs[0].set_ylabel(r"$s_j$")

axs[0].set_yticks([0, 0.2, 0.4,])
y_pos = [text.get_position()[1] for text in axs[0].get_yticklabels()]
y_txt = [text.get_text() for text in axs[0].get_yticklabels()]

new_pos0 = 1/np.sqrt(gamma_0)
new_txt0 = r"$1/\sqrt{\gamma_l}$"
new_pos1 = 1/np.sqrt(gamma_1)
new_txt1 = r"$1/\sqrt{\gamma_s}$"

y_pos.append(new_pos0)
y_pos.append(new_pos1)
y_txt.append(new_txt0)
y_txt.append(new_txt1)
axs[0].set_yticks(y_pos,y_txt)

#t-ds_j/dt
for i in reversed(range(2)):
    axs[1].plot(np.array(dt(weight_singular[i])) / lr_simple, alpha=0.5, linewidth=6, color=colors[i], label=(r"$\frac{ds_{" + ("l" if i == 0 else "s") +"}" + r"}{dt}$"))
axs[1].set_xlim(0, epochs)
axs[1].hlines([np.sqrt(gamma_0 * 2)], xmin=-1, xmax=epochs*1.1, color=colors[0], linewidth=3, alpha=0.8, linestyle='--')
axs[1].hlines([np.sqrt(gamma_1 * 2)], xmin=-1, xmax=epochs*1.1, color=colors[1], linewidth=3, alpha=0.8, linestyle='--')

axs[1].set_yticks([0, 6, 12])

y_pos = [text.get_position()[1] for text in axs[1].get_yticklabels()]#[:-1]
y_txt = [text.get_text() for text in axs[1].get_yticklabels()]#[:-1]

new_pos0 = np.sqrt(gamma_0 * 2)
new_txt0 = r"$\sqrt{2 \gamma_l}$"
new_pos1 = np.sqrt(gamma_1 * 2)
new_txt1 = r"$\sqrt{2 \gamma_s}$"

y_pos.append(new_pos0)
y_pos.append(new_pos1)
y_txt.append(new_txt0)
y_txt.append(new_txt1)
axs[1].set_yticks(y_pos,y_txt)

axs[1].set_xlabel("Step")
axs[1].set_ylabel(r"$\frac{ds_j}{dt}$")
axs[0].legend()
axs[1].legend()
plt.tight_layout()
plt.savefig("figure/freq_s_ds_dt.pdf", format="pdf", bbox_inches = 'tight')
plt.show()

In [ ]:
#||A^TB||^2/2
fig, axs = plt.subplots(1, 1, figsize=(12, 4))
axs.plot(plane_alignment, alpha=0.8, color=colors[2], linewidth=3)
axs.set_xlim(0, epochs)
axs.set_ylim(-0.05, 1.05)
axs.set_xlabel("Step")
axs.set_ylabel("Subspace Alignment")
plt.savefig("figure/freq_alignment.pdf", format="pdf", bbox_inches = 'tight')
plt.show()
print(max(plane_alignment))

In [ ]:
#3d plot x: u1, y: u2, z: step, line1: We1, line2: We2
#use uwe[i][j], as uwe[0] means x, uwe[1] means z

def reject_outliers(data, m=0.15):
    indexes = np.where(abs(np.array(dt(data))) > m)
    for i in indexes:
        r = 10
        for j in range(-r, r):
            data[i+j] = data[i+j-1]
    return data

fig = plt.figure(figsize=(22, 10), constrained_layout=True)
ax = fig.add_subplot(projection='3d')
for i in reversed(range(2)):
    ax.plot(np.arange(0, epochs, 1), reject_outliers(np.abs(uwe[0][i])), reject_outliers(np.abs(uwe[1][i])), alpha=0.5, linewidth=5,color=colors[i], label=(r"$\text{Proj}_{U^{(\leq 2)}}(We_" + eig_order[i] + ")$"))
ax.plot(np.arange(0, epochs, 1), np.sqrt(y1), 0, color=colors[1], linestyle='--')
ax.plot(np.arange(0, epochs, 1), 0, np.sqrt(y2), color=colors[0], linestyle='--')
ax.set_xlabel("Step", labelpad=6)
ax.set_ylabel(r"$u_1$", labelpad=6)
ax.set_zlabel(r"$u_2$", labelpad=6)
ax.set_box_aspect(aspect=None, zoom=0.99)
plt.legend(loc='upper right', bbox_to_anchor=(1, 0.9))
plt.savefig("figure/freq_3d.pdf", format="pdf", bbox_inches = 'tight')
plt.show()